# KWISMO — Collecte de données (scraping + OCR)

Ce notebook n'écrit aucune logique lui-même : il appelle uniquement le code de `src/data/` (scrape.py, scrape_social.py, ocr.py, metrics.py). Le projet exige **Python 3.13** (voir `src/__init__.py`).

Ce notebook s'adapte automatiquement à votre environnement (**Local**, **Google Colab** ou **Kaggle Notebooks**).

In [ ]:
# Détection automatique de l'environnement (Local, Google Colab, Kaggle Notebooks)
import os
import sys
import subprocess

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB:
    ENV_NAME = "Google Colab"
elif ON_KAGGLE:
    ENV_NAME = "Kaggle Notebooks"
else:
    ENV_NAME = "Local"

print(f"Environnement de calcul détecté : {ENV_NAME}")

In [ ]:
# Gestion du répertoire de travail et clônage du dépôt sur les plateformes Cloud
from pathlib import Path

if ON_COLAB:
    PROJECT_DIR = Path("/content/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /content/kwismo
    else:
        !git -C /content/kwismo fetch && git -C /content/kwismo reset --hard origin/main
    os.chdir(PROJECT_DIR)
elif ON_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /kaggle/working/kwismo
    else:
        !git -C /kaggle/working/kwismo fetch && git -C /kaggle/working/kwismo reset --hard origin/main
    os.chdir(PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd()
    if (PROJECT_DIR / "kwismo-ai").exists():
        PROJECT_DIR = PROJECT_DIR / "kwismo-ai"
        os.chdir(PROJECT_DIR)

print("Dossier de travail :", PROJECT_DIR)

## Installation de Python 3.13 isolé et des dépendances (Colab / Kaggle)

Sur les environnements Cloud, télécharge et installe **Python 3.13**, crée un venv `.venv313`, installe `requirements.txt` et Playwright Chromium.

In [ ]:
VENV_DIR = PROJECT_DIR / ".venv313"

if ON_COLAB or ON_KAGGLE:
    python313_bin = VENV_DIR / "bin" / "python"
    if not python313_bin.exists():
        print("⚡ Installation de Python 3.13 et création de l'environnement .venv313...")
        !apt-get update -y
        !apt-get install -y software-properties-common
        !add-apt-repository -y ppa:deadsnakes/ppa
        !apt-get update -y
        !apt-get install -y python3.13 python3.13-venv python3.13-dev
        !python3.13 -m venv {VENV_DIR}
        !{VENV_DIR}/bin/pip install --upgrade pip
        !{VENV_DIR}/bin/pip install -r requirements.txt
        !{VENV_DIR}/bin/python -m playwright install --with-deps chromium
    PYTHON_BIN = str(python313_bin)
else:
    PYTHON_BIN = sys.executable

ver_proc = subprocess.run([PYTHON_BIN, "--version"], capture_output=True, text=True)
print("Interprète Python configuré :", PYTHON_BIN)
print("Version vérifiée :", ver_proc.stdout.strip() or ver_proc.stderr.strip())

## Configuration (`.env`)

Création du fichier d'environnement minimal si nécessaire.

In [ ]:
if ON_COLAB or ON_KAGGLE:
    env_lines = [
        'MODEL_DIR="./models"',
        'HF_MODEL_NAME="Davlan/afro-xlmr-base"',
        '# FACEBOOK_ACCOUNTS="user1:pass1,user2:pass2"  # décommenter si besoin',
    ]
    Path(".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
    print(".env créé pour la session cloud.")
else:
    print("Local : .env configuré.")

In [ ]:
# Lancement d'un module Python src/ avec Python 3.13 en sous-processus
def run_module(module: str) -> None:
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a échoué (code {result.returncode})")
    else:
        print(result.stdout)

## 1. Sources texte (découverte dynamique via recherche web)

In [ ]:
# Recherche de pages via mots-clés, extraction texte + images, dédoublonnage
run_module("src.data.scrape")

## 2. Facebook / Instagram / X (si comptes configurés)

In [ ]:
# Connexion aux comptes et collecte sociale (optionnel)
run_module("src.data.scrape_social")

## 3. OCR des captures d'écran collectées

In [ ]:
# Extraction du texte des images via EasyOCR avec Python 3.13
run_module("src.data.ocr")

## 4. Métriques de la collecte

In [ ]:
# Mise à jour des métriques et des graphiques d'évolution
run_module("src.data.metrics")

In [ ]:
# Affichage des graphiques d'évolution
from IPython.display import Image, display

plots_dir = PROJECT_DIR / "data" / "interim" / "metrics" / "plots"
if (plots_dir / "evolution_collecte.png").exists():
    display(Image(filename=str(plots_dir / "evolution_collecte.png")))
if (plots_dir / "erreurs_par_run.png").exists():
    display(Image(filename=str(plots_dir / "erreurs_par_run.png")))

## 5. Sauvegarde / Publication des résultats

Les données sont stockées dans `data/raw/scraped/messages.jsonl`.

In [ ]:
SAVE_METHOD = "drive"  # Choisir "drive" ou "git"

if ON_COLAB and SAVE_METHOD == "drive":
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/kwismo_data")
    dest.mkdir(parents=True, exist_ok=True)
    if Path("data/raw/scraped/messages.jsonl").exists():
        shutil.copy("data/raw/scraped/messages.jsonl", dest / "messages.jsonl")
    if Path("data/interim/known_domains.json").exists():
        shutil.copy("data/interim/known_domains.json", dest / "known_domains.json")
    if Path("data/raw/scraped/images").exists():
        shutil.copytree("data/raw/scraped/images", dest / "images", dirs_exist_ok=True)
    if Path("data/interim/metrics").exists():
        shutil.copytree("data/interim/metrics", dest / "metrics", dirs_exist_ok=True)
    print("Sauvegarde sur Google Drive terminée avec succès :", dest)
elif SAVE_METHOD == "git":
    !git config user.email "toi@exemple.com"
    !git config user.name "Ton Nom"
    !git add data/raw/scraped/*.jsonl data/interim/metrics/ data/interim/known_domains.json
    !git commit -m "Collecte de données (scraping session)"
    print("Données committées sur Git.")
else:
    print("Collecte achevée. Données sauvegardées localement dans kwismo-ai/data/raw/scraped/")